In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from robusta_hmf import Robusta
import time
#%matplotlib widget

In [ ]:
#define constants
c = 299792.458 # km/s
LN10 = np.log(10.)
MAX_IVAR = 2.5e3
MIN_IVAR = 0.1
H_ALPHA, H_BETA = 6564.614, 4862.721 #Reference: from classic.sdss.org
HE_I = 6680 #Reference: Johanna brain
HE_II = 4686 #Reference: Johanna brain

In [ ]:
#read in the data and deal with radial velocities
with open('spectra_5021_2026-07-15_data.pkl','rb') as f:
     fluxes, loglam, ivars, continuua, spec_files, rvs, ews_halphas, nana_hash, parent_spectra = pickle.load(f)
print(fluxes.shape, loglam.shape, ivars.shape, continuua.shape, len(spec_files), rvs.shape, ews_halphas.shape, nana_hash.shape, parent_spectra.shape)

In [ ]:
print(spec_files[:5])

In [ ]:
# remove spectra where Halpha equivalent width is negative (emission line)
good_ew_mask = ews_halphas >= 0

print(f"Keeping {good_ew_mask.sum()} / {len(good_ew_mask)} spectra (removed {(~good_ew_mask).sum()} with negative EW)")

fluxes = fluxes[good_ew_mask]
ivars = ivars[good_ew_mask]
continuua = continuua[good_ew_mask]
rvs = rvs[good_ew_mask]
ews_halphas = ews_halphas[good_ew_mask]
nana_hash = nana_hash[good_ew_mask]
spec_files = [sf for sf, keep in zip(spec_files, good_ew_mask) if keep]
parent_spectra = parent_spectra[good_ew_mask]

print(fluxes.shape, ivars.shape, continuua.shape, len(spec_files), rvs.shape, ews_halphas.shape, nana_hash.shape, parent_spectra.shape)

In [ ]:
#load stellar parameters

df = pd.read_parquet("parent_stars_2026-07-15.parquet")
print("num of stars in parquet:", df.shape[0])

# print(df.columns.to_list())

In [ ]:
#log lambda values have been corrupted
print(10**loglam)
print(loglam[1:] - loglam[:-1])
print(np.median(loglam[1:] - loglam[:-1]))
delta_log_lambda_pixel = np.median(loglam[1:] - loglam[:-1])
wave = 10**loglam

In [ ]:
#get ready to do pixel shifts
delta_log_lambdas = (rvs / c) / LN10
delta_pixels = np.round(delta_log_lambdas/delta_log_lambda_pixel).astype(int)

print(delta_log_lambdas, delta_pixels)

In [ ]:
#shift spectra to the rest frame
## johanna is not going to like this

rest_fluxes = np.zeros_like(fluxes) + 1.
print(rest_fluxes.shape, fluxes.shape, data.shape)
rest_ivars = np.zeros_like(ivars)
rest_continuua = np.zeros_like(continuua) + np.nan

for i, dp in enumerate(delta_pixels):
    if dp < 0:
        rest_fluxes[i, -dp:] = fluxes[i, :dp]
        rest_ivars[i, -dp:] = ivars[i, :dp]
        rest_continuua[i, -dp:] = continuua[i, :dp]
        
    elif dp > 0:
        rest_fluxes[i, :-dp] = fluxes[i, dp:]
        rest_ivars[i, :-dp] = ivars[i, dp:]
        rest_continuua[i, :-dp] = continuua[i, dp:]
        
    else:
        rest_fluxes[i, :] = fluxes[i, :]
        rest_ivars[i, :] = ivars[i, :]
        rest_continuua[i, :] = continuua[i, :]


In [ ]:
#fix nans and infinities
bad = np.logical_not(np.isfinite(rest_fluxes))
rest_fluxes[bad] = 1
rest_ivars[bad] = 0

bad = np.logical_not(np.isfinite(rest_ivars))
rest_fluxes[bad] = 1
rest_ivars[bad] = 0

bad = np.logical_or((rest_fluxes > 2.0), (rest_fluxes < 0))
rest_fluxes[bad] = 1
rest_ivars[bad] = 0

bad = rest_ivars > MAX_IVAR
# rest_fluxes[bad] = 1
# rest_ivars[bad] = 0
rest_ivars[bad] = MAX_IVAR

bad = rest_ivars < MIN_IVAR
rest_fluxes[bad] = 1.0
rest_ivars[bad] = MIN_IVAR

In [ ]:
#we are using the rest frame data as our data 
data = rest_fluxes
weights = rest_ivars

In [ ]:
# split data to A and B using nana_hash parity, only train on A
N, M = fluxes.shape
print(fluxes.shape)

Aindx = np.where(nana_hash % 2 == 0)[0]
Bindx = np.where(nana_hash % 2 == 1)[0]

print(f"A: {len(Aindx)} spectra, B: {len(Bindx)} spectra")

# hyperparameters set here
K = 12
modelA = Robusta(rank=K, robust=True, robust_scale=1, robust_nu=1)
start = time.perf_counter()
modelA.fit(data[Aindx], weights[Aindx], max_iter=10000)
end = time.perf_counter()
print("total time:", (end - start)/60, "minutes")

In [ ]:
# synthesize, but masking Balmer lines.
log_H_ALPHA, log_H_BETA, log_HE_I, log_HE_II = np.log10(H_ALPHA), np.log10(H_BETA), np.log10(HE_I), np.log10(HE_II)

H_delta_lnlam_line = 1000 / c # km/s, this is the v/c ratio notice must be natural log for ln(x + 1) ≈ x to hold
H_delta_loglam_line = H_delta_lnlam_line / LN10 #convert to log10 units must match!!
region_alpha = np.abs(loglam - log_H_ALPHA)
region_beta = np.abs(loglam - log_H_BETA)

#do the same but half width of 500km/s for helium lines
HE_delta_lnlam_line = 500 / c
HE_delta_loglam_line = HE_delta_lnlam_line / LN10
region_he_i = np.abs(loglam - log_HE_I)
region_he_ii = np.abs(loglam - log_HE_II)


censor_mask = np.ones_like(loglam)
censor_mask[(region_alpha < H_delta_loglam_line) | (region_beta < H_delta_loglam_line) 
          | (region_he_i < HE_delta_loglam_line) | (region_he_ii < HE_delta_loglam_line)] = 0 #?

state, _ = modelA.infer(data[Bindx], weights[Bindx] * censor_mask[None, :])
synth_B = modelA.synthesize(state)

In [ ]:
# check that mask is not wrong
plt.plot(10**loglam, censor_mask, 'k')
plt.axvline(H_ALPHA, alpha = 0.5, color = 'violet')
plt.axvline(H_BETA, alpha = 0.5, color = 'violet')
plt.axvline(HE_I, alpha = 0.5)
plt.axvline(HE_II, alpha = 0.5)
plt.xlim(4200,7000)
plt.show()

## Eigenspectra Plotting

In [ ]:
eigsA = modelA.basis_vectors().T

f = plt.figure(figsize=(15, 15))
plt.title("Eigenspectra")
offset = 0.2
for i in range(K):
    f.gca().plot(wave, eigsA[i] + i * offset)

#plt.axvline(H_ALPHA, lw=1, alpha=0.5, color="blue")
plt.xlabel("Wavelength (angstrom)")
plt.show()

## Synthesized Spectra Examples

In [ ]:
# look at a few spectra and syntheses
f = plt.figure(figsize=(15, 15))
offset = 0.5
for i in range(12):
    f.gca().plot(wave, data[Bindx[i]] + i * offset, color="k")
    f.gca().plot(wave, synth_B[i] + i * offset, color="r")

plt.axvline(H_ALPHA, lw=1, alpha=0.5, color="b")
plt.axvline(H_BETA, lw=1, alpha=0.5, color="b")
plt.axvline(HE_I, lw=1, alpha=0.5, color="b")
plt.axvline(HE_II, lw=1, alpha=0.5, color="b")
plt.xlim(4200, 7000)
plt.xlabel("Wavelength (angstrom)")
plt.show()

## Residual Examples

In [ ]:
# look at residuals at H-alpha
f = plt.figure(figsize=(15, 15))
offset = 0.5
for i in range(12):
    f.gca().plot(wave, data[Bindx[i]] + i * offset - synth_B[i], color="k")

plt.axvline(H_ALPHA, lw=1, alpha=0.5, color="b")
plt.axvline(H_BETA, lw=1, alpha=0.5, color="b")
plt.xlabel("Wavelength (angstrom)")
#plt.xlim(H_ALPHA - 200., H_ALPHA + 200.)
plt.title("Residual examples")
plt.axvline(H_ALPHA, lw=1, alpha=0.5, color="b")
plt.axvline(H_BETA, lw=1, alpha=0.5, color="b")
plt.axvline(HE_I, lw=1, alpha=0.5, color="b")
plt.axvline(HE_II, lw=1, alpha=0.5, color="b")
plt.xlim(4200, 7000)
plt.show()

## measure equivalent widths


Recall how we defined some stuff in the censor masking section  <br>
H_delta_lnlam_line = 1000 / c #1000km/s half width for H_ALPHA, H_BETA  <br>
H_delta_loglam_line = H_delta_lnlam_line / LN10   <br>
HE_delta_lnlam_line = 500 / c #500km/s half width for HE_I, HE_II  <br>
HE_delta_loglam_line = HE_delta_lnlam_line / LN10 <br>
 <br>
psuedo code from hogg for additional centroid, variance, and moment4 calculation <br>
now imagine that we have lam = 10 **loglam, lam0 = line center (like halpha) <br>
centroid = sum(resid *(lam - lam0) * integration weights) <br>
variance = sum(resid * (lam - lam0) ** 2 * integration weights) <br>
moment4 = sum(resid * (lam-lam0) ** 4 * integration weights) <br>
ews = sum(resid * integration_weight) <br>
 <br>
now each of these integrals has the form xx = sum(resid * something) <br>

**ASK HOGG**: earlier we set weights = rest_ivars and for the ivar in the err calucation here <br>
I'm going to using weights[Bindx] hopefully thats chill <br>
**ASK HOGG** I also added a np.abs before the sqrt

the uncertainty xx is (formally): xx_err = sqrt(sum(something**2)/ivar)) <br>

for your code you should produce EW, EW_err, var, var_err, etc <br>

In [ ]:
ivar = weights[Bindx]
print(ivar.shape)
print(np.all(ivar)) #confirms that ivar has 0 values :( why?
print(np.all(weights[Bindx]))

In [ ]:
lam = 10 ** loglam
N, M = data[Bindx].shape
lines = [H_ALPHA, H_BETA, HE_I, HE_II]
line_labels = ["Halpha", "Hbeta", "HeI", "HeII"]

#make sure half_widths and lines match!!!!!!
half_wids = [H_delta_loglam_line, H_delta_loglam_line, HE_delta_loglam_line, HE_delta_loglam_line] 

# get ready to take moments
exponents = np.arange(5)
moment_names = ["EW", "centroid", "variance", "skew", "m4"]
moments = np.zeros((N, len(lines), len(exponents))) + np.nan
moment_errs = np.zeros((N, len(lines), len(exponents))) + np.nan

# take all moments of all lines
for j, (line, wid) in enumerate(zip(lines, half_wids)):
    logline = np.log10(line)
    integration_weight = delta_log_lambda_pixel * LN10 * lam * (np.abs(logline - loglam) < wid)
    resid = data[Bindx] - synth_B
    ivar = weights[Bindx]
    diff = lam - line #this is hogg's lam-lam0 he wrote about

    #compute vals and corresponding errors
    for k, expo in enumerate(exponents):
        something = integration_weight * diff ** expo
        moments[:, j, k] = np.sum(resid * something[None, :], axis=1)
        moment_errs[:, j, k] = np.sqrt(np.sum(something[None, :] ** 2 / ivar, axis=1))
    
print(moments)
print(moment_errs)

## Scatter Plots of EWS

In [ ]:
# NANA: Make the error bars look not horrible.
def minmax(q):
    return np.min(q), np.max(q)
sick = moment_errs[:, 0, 0] < 1.0
for j in [1, 2, 3]:
    plt.errorbar(moments[sick, 0, 0], moments[sick, j, 0],
                 c="k", marker=".", ls="none", alpha=0.25,
                 xerr = moment_errs[sick, 0, 0], yerr = moment_errs[sick, j, 0])
    plt.xlabel(line_labels[0] + " " + moment_names[0])
    plt.ylabel(line_labels[j] + " " + moment_names[0])
    plt.xlim(minmax(moments[sick, 0, 0]))
    plt.ylim(minmax(moments[sick, j, 0]))
    plt.show()

## Sorted by highest EW, Residuals

In [ ]:
#sort the synthesized 
foo = np.argsort(moments[:,0, 0] / moment_errs[:, 0, 0]) #[::-1]
f = plt.figure(figsize=(10, 8))
offset = 0.5
for i in range(12):
    f.gca().plot(wave, data[Bindx[foo[i]]] + i * offset - synth_B[foo[i]], color="k")

plt.axvline(H_ALPHA, lw=1, alpha=0.5, color="b")
plt.axvline(H_BETA, lw=1, alpha=0.5, color="b")
plt.xlabel("Wavelength (angstrom)")
plt.xlim(H_ALPHA - 200., H_ALPHA + 200.)
halpha_lo = 10**(log_H_ALPHA - H_delta_loglam_line)
halpha_hi = 10**(log_H_ALPHA + H_delta_loglam_line)
plt.axvspan(halpha_lo, halpha_hi, color='royalblue', alpha=0.3)
hbeta_lo = 10**(log_H_BETA - H_delta_loglam_line)
hbeta_hi = 10**(log_H_BETA + H_delta_loglam_line)
plt.axvspan(hbeta_lo, hbeta_hi, color='royalblue', alpha=0.3)
plt.title("Residual at Halpha")
plt.savefig("resid_halpha.png")
plt.show()

## Sorted by highest EW, synthesized spectra

In [ ]:
# look at a few spectra and syntheses

f = plt.figure(figsize=(10, 8))
offset = 0.5
ax = f.gca()
for i in range(12):
    ax.plot(wave, data[Bindx[foo[i]]] + i * offset, color="k")
    ax.plot(wave, synth_B[foo[i]] + i * offset, color="r")
    #fname = spec_files[Bindx[foo[i]]]
    #ax.text(wave[-1], i * offset + 1, fname, fontsize=6, ha='right', va='center')
    

# f = plt.figure(figsize=(10, 8))
# offset = 0.5
# for i in range(12):
#     f.gca().plot(wave, data[Bindx[foo[i]]] + i * offset, color="k")
#     f.gca().plot(wave, synth_B[foo[i]] + i * offset, color="r")
#     fname = spec_files[Bindx[foo[i]]]
#     print("filename:", fname)
#     plt.text(wave[-1], i * offset + 1, fname, fontsize=6, ha='right', va='center')

# f = plt.figure(figsize=(10, 8))
# ax = f.gca()
# offset = 0.5
# for i in range(12):
#     ax.plot(wave, data[Bindx[foo[i]]] + i * offset, color="k")
#     ax.plot(wave, synth_B[foo[i]] + i * offset, color="r")
#     fname = spec_files[Bindx[foo[i]]]
#     ax.text(1.01, (i * offset + 1 - ax.get_ylim()[0]) / (ax.get_ylim()[1] - ax.get_ylim()[0]),
#             fname, fontsize=6, ha='left', va='center', transform=ax.transAxes,
#             bbox=dict(facecolor='white', edgecolor='none', alpha=0.7, pad=1))

plt.axvline(H_ALPHA, lw=1, alpha=0.5, color="b")
plt.axvline(H_BETA, lw=1, alpha=0.5, color="b")
plt.axvline(HE_II, lw=1, alpha=0.5, color="b")
plt.xlabel("Wavelength (angstrom)")
plt.xlim(H_ALPHA - 500, H_ALPHA + 500) 
halpha_lo = 10**(log_H_ALPHA - H_delta_loglam_line)
halpha_hi = 10**(log_H_ALPHA + H_delta_loglam_line)
plt.axvspan(halpha_lo, halpha_hi, color='royalblue', alpha=0.3)
plt.title("Synthsized spectra at Halpha")
#plt.savefig("synth_halpha.png")
plt.show()

In [ ]:
# look at a few spectra and syntheses
f = plt.figure(figsize=(10, 10))
offset = 0.5
for i in range(12):
    f.gca().plot(wave, data[Bindx[foo[i]]] + i * offset, color="k")
    f.gca().plot(wave, synth_B[foo[i]] + i * offset, color="r")

plt.axvline(H_ALPHA, lw=1, alpha=0.5, color="b")
plt.axvline(H_BETA, lw=1, alpha=0.5, color="b")
plt.axvline(HE_II, lw=1, alpha=0.5, color="b")
plt.xlabel("Wavelength (angstrom)")
#plt.xlim(H_ALPHA - 200, H_ALPHA + 200) 
halpha_lo = 10**(log_H_ALPHA - H_delta_loglam_line)
halpha_hi = 10**(log_H_ALPHA + H_delta_loglam_line)
plt.axvspan(halpha_lo, halpha_hi, color='royalblue', alpha=0.3)
plt.title("Synthsized spectra at Halpha")
#plt.savefig("synth_halpha.png")
plt.show()